In [1]:
import torch, time, threading



# 50% utilization on l40s
SIZE = 4096             # smaller on L40S
DTYPE = torch.float16
BURST_ITERS = 30         # do 1–4 GEMMs per cycle
SLEEP_MS = 12           # increase to reduce util; decrease to increase util

assert torch.cuda.is_available()
a = torch.randn(SIZE, SIZE, device="cuda", dtype=DTYPE)
b = torch.randn(SIZE, SIZE, device="cuda", dtype=DTYPE)

stop_flag = False

def burn():
    global stop_flag
    # warmup
    for _ in range(5):
        _ = a @ b
    torch.cuda.synchronize()

    sleep_s = SLEEP_MS / 1000.0

    while not stop_flag:
        # burst: enqueue a small, bounded amount of work
        for _ in range(BURST_ITERS):
            _ = a @ b

        # drain queue so sleep actually idles
        torch.cuda.synchronize()
        time.sleep(sleep_s)

thread = threading.Thread(target=burn, daemon=True)
thread.start()
print("Background load started.")


Background load started.


In [2]:
stop_flag = True
thread.join()
print("Stopped.")

Stopped.
